[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_5_aruco/13_detect_aruco.ipynb)

# Notebook 13 — Detecting ArUco Markers

**Part 5: ArUco Markers** | Estimated time: 40 min

---

We have markers. Now we need to **find them** in an image.

```
Pipeline position:

  Generate marker  →  Print  →  [Detect corners + ID]  →  Pose estimate
                                        ← YOU ARE HERE
```

Detection gives us:
- **4 corner points** per marker (sub-pixel accurate)
- **Marker ID** (which dictionary entry it matches)
- **Rejected candidates** (regions that looked like markers but failed validation)

## Recommended Watch

> Watch this **before** opening the notebook — it shows ArUco detection with an AR overlay, giving good visual context for what detection looks like in practice.

| Title | Link | Duration |
|---|---|---|
| **Building an Augmented Reality Application with ArUco Marker Detection in OpenCV** | [▶ Watch](https://www.youtube.com/watch?v=UlM2bpqo_o0) | ~20 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python matplotlib numpy -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Understand what `corners`, `ids`, and `rejected` mean
- Call `detectMarkers` (old API) and `ArucoDetector.detectMarkers` (new API)
- Draw detection results on images
- Interpret why a marker might fail to be detected
- Use `DetectorParameters` to tune detection sensitivity
- Write a real-time webcam detection loop

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'OpenCV version: {cv2.__version__}')

def show_bgr(img, title='', figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_gray(img, title='', figsize=(5, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 1. The Detection API (Old vs. New)

### Old API (OpenCV ≤ 4.6)

```python
arucoDict   = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
arucoParams = cv2.aruco.DetectorParameters_create()
corners, ids, rejected = cv2.aruco.detectMarkers(
    gray, arucoDict, parameters=arucoParams
)
```

### New API (OpenCV ≥ 4.7)

```python
arucoDict    = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
arucoParams  = cv2.aruco.DetectorParameters()
detector     = cv2.aruco.ArucoDetector(arucoDict, arucoParams)
corners, ids, rejected = detector.detectMarkers(gray)
```

The return values are the same in both APIs.

### What Each Output Means

| Output | Shape | Meaning |
|---|---|---|
| `corners` | `list[ndarray(1,4,2)]` | 4 (x,y) corner points per marker |
| `ids` | `ndarray(N,1)` or `None` | ID of each detected marker |
| `rejected` | `list[ndarray(1,4,2)]` | Candidates that failed validation |

In [ ]:
# Compatibility shim — works with both old and new API
def make_detector(dict_id):
    """Returns (detector_or_dict, params_or_None) depending on API version."""
    try:
        # New API
        d = cv2.aruco.getPredefinedDictionary(dict_id)
        p = cv2.aruco.DetectorParameters()
        detector = cv2.aruco.ArucoDetector(d, p)
        return ('new', detector, d, p)
    except AttributeError:
        # Old API
        d = cv2.aruco.Dictionary_get(dict_id)
        p = cv2.aruco.DetectorParameters_create()
        return ('old', None, d, p)

def detect_markers(image, api_info):
    """Detect ArUco markers — API-agnostic."""
    api_ver, detector, arucoDict, arucoParams = api_info
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    if api_ver == 'new':
        return detector.detectMarkers(gray)
    else:
        return cv2.aruco.detectMarkers(gray, arucoDict, parameters=arucoParams)

# Create detector for DICT_4X4_50
api_info = make_detector(cv2.aruco.DICT_4X4_50)
print(f'Using {api_info[0]} ArUco API')

## 2. Creating a Test Image

We'll create a synthetic test image with multiple markers at different positions and orientations.  
This lets us run the full detection pipeline without needing a camera.

In [ ]:
def create_test_scene(marker_ids=[0, 1, 2], scene_size=800, marker_px=150, noise_std=5):
    """
    Create a synthetic scene with multiple ArUco markers at random positions.
    
    Args:
        marker_ids: list of marker IDs to place
        scene_size:  image size (square)
        marker_px:   marker pixel size
        noise_std:   Gaussian noise standard deviation (0 = no noise)
    """
    aruco_dict_obj = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50) \
        if hasattr(cv2.aruco, 'getPredefinedDictionary') \
        else cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
    
    # Gray background simulating a wall/floor
    scene = np.ones((scene_size, scene_size, 3), dtype=np.uint8) * 200
    # Add texture
    np.random.seed(42)
    scene += np.random.randint(-20, 20, scene.shape, dtype=np.int8).view(np.uint8) \
             .reshape(scene.shape)
    scene = np.clip(scene, 0, 255).astype(np.uint8)
    
    placements = [
        {'center': (180, 200), 'angle': 0},
        {'center': (550, 180), 'angle': 15},
        {'center': (350, 560), 'angle': -10},
    ]
    
    for mid, placement in zip(marker_ids, placements[:len(marker_ids)]):
        # Generate marker
        try:
            m = cv2.aruco.generateImageMarker(aruco_dict_obj, mid, marker_px)
        except AttributeError:
            buf = np.zeros((marker_px, marker_px, 1), dtype='uint8')
            cv2.aruco.drawMarker(aruco_dict_obj, mid, marker_px, buf, 1)
            m = buf.squeeze()
        m_bgr = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        
        # Add slight rotation if requested
        angle = placement['angle']
        if angle != 0:
            h, w = m_bgr.shape[:2]
            rot_mat = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
            m_bgr = cv2.warpAffine(m_bgr, rot_mat, (w, h),
                                    borderMode=cv2.BORDER_CONSTANT,
                                    borderValue=(200, 200, 200))
        
        # Paste into scene
        cx, cy = placement['center']
        h, w = m_bgr.shape[:2]
        x0, y0 = cx - w//2, cy - h//2
        x1, y1 = x0 + w, y0 + h
        
        # Clip to scene bounds
        sx0 = max(0, x0); sy0 = max(0, y0)
        sx1 = min(scene_size, x1); sy1 = min(scene_size, y1)
        mx0 = sx0 - x0; my0 = sy0 - y0
        mx1 = mx0 + (sx1 - sx0); my1 = my0 + (sy1 - sy0)
        
        scene[sy0:sy1, sx0:sx1] = m_bgr[my0:my1, mx0:mx1]
    
    # Add Gaussian noise
    if noise_std > 0:
        noise = np.random.normal(0, noise_std, scene.shape).astype(np.int16)
        scene = np.clip(scene.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return scene

# Create test scene
test_scene = create_test_scene(marker_ids=[0, 1, 2])
show_bgr(test_scene, 'Synthetic test scene — 3 ArUco markers (IDs 0, 1, 2)')
print(f'Scene shape: {test_scene.shape}')

## 3. Running Detection

Detection works on **grayscale** images. Color adds no information and only slows things down.

In [ ]:
# Detect markers
corners, ids, rejected = detect_markers(test_scene, api_info)

# --- Interpret the results ---
print('=== Detection Results ===')

if ids is not None:
    flat_ids = ids.flatten()
    print(f'Detected {len(flat_ids)} markers')
    for i, (corner, mid) in enumerate(zip(corners, flat_ids)):
        pts = corner.reshape(4, 2)
        tl, tr, br, bl = pts  # ArUco corner order
        center = pts.mean(axis=0)
        print(f'  Marker ID={mid}: center=({center[0]:.0f}, {center[1]:.0f})')
        print(f'    TL={tl.astype(int)}, TR={tr.astype(int)}')
        print(f'    BR={br.astype(int)}, BL={bl.astype(int)}')
else:
    print('No markers detected!')

print(f'\nRejected candidates: {len(rejected)}')
print('(These looked like markers but failed ID validation)')

### Corner Order Convention

ArUco always returns corners in this exact order:

```
corners[i] shape: (1, 4, 2)

After reshape(4, 2):
  [0] = top-left
  [1] = top-right
  [2] = bottom-right
  [3] = bottom-left

Viewing the marker face-on:
  TL ───── TR
  │         │
  BL ───── BR
```

This is critical for pose estimation — the convention tells OpenCV which corner corresponds to which corner of the known 3D square.

## 4. Drawing Detection Results

Detection gives you data: corner arrays and IDs. To actually see what was detected — and to debug when something goes wrong — you draw those results back onto the image.

A useful two-layer approach: first draw the **corners** precisely (to verify the detected positions are accurate), then draw the **IDs** as text overlays (to verify the right marker was identified and there are no false positives). When debugging failed detections, the corners layer tells you whether the problem is in corner localization or in ID decoding.

In [ ]:
def draw_detections(image, corners, ids, rejected=None, draw_rejected=True):
    """
    Draw detected markers and optionally rejected candidates.
    
    Returns annotated image copy.
    """
    output = image.copy()
    
    if ids is not None:
        flat_ids = ids.flatten()
        
        for corner_set, mid in zip(corners, flat_ids):
            pts = corner_set.reshape(4, 2).astype(int)
            tl, tr, br, bl = pts
            
            # Draw bounding box (green)
            cv2.line(output, tuple(tl), tuple(tr), (0, 255, 0), 2)
            cv2.line(output, tuple(tr), tuple(br), (0, 255, 0), 2)
            cv2.line(output, tuple(br), tuple(bl), (0, 255, 0), 2)
            cv2.line(output, tuple(bl), tuple(tl), (0, 255, 0), 2)
            
            # Draw corners (different color per corner)
            corner_colors = [(0,0,255), (255,0,0), (0,255,255), (255,0,255)]  # TL TR BR BL
            corner_names  = ['TL', 'TR', 'BR', 'BL']
            for pt, color, name in zip(pts, corner_colors, corner_names):
                cv2.circle(output, tuple(pt), 6, color, -1)
            
            # Draw center dot
            cx = int((tl[0] + br[0]) / 2)
            cy = int((tl[1] + br[1]) / 2)
            cv2.circle(output, (cx, cy), 4, (255, 255, 0), -1)
            
            # Draw ID text
            cv2.putText(output, f'ID={mid}',
                        (tl[0], tl[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (0, 255, 0), 2)
    
    # Draw rejected candidates in red (optional, useful for debugging)
    if draw_rejected and rejected is not None:
        for corner_set in rejected:
            pts = corner_set.reshape(4, 2).astype(int)
            tl, tr, br, bl = pts
            cv2.line(output, tuple(tl), tuple(tr), (0, 0, 255), 1)
            cv2.line(output, tuple(tr), tuple(br), (0, 0, 255), 1)
            cv2.line(output, tuple(br), tuple(bl), (0, 0, 255), 1)
            cv2.line(output, tuple(bl), tuple(tl), (0, 0, 255), 1)
    
    # Status text
    n = len(ids.flatten()) if ids is not None else 0
    cv2.putText(output, f'Detected: {n} markers',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255, 255, 255), 2)
    
    return output

# Draw results
annotated = draw_detections(test_scene, corners, ids, rejected, draw_rejected=True)
show_bgr(annotated, 'Detection results (green=detected, red=rejected candidates)',
         figsize=(9, 9))

### OpenCV's Built-in Drawing Function

OpenCV also provides `cv2.aruco.drawDetectedMarkers()` which is simpler to call:

In [ ]:
# Using the built-in drawing function
annotated_builtin = test_scene.copy()

if ids is not None:
    cv2.aruco.drawDetectedMarkers(
        annotated_builtin,  # image (modified in-place)
        corners,            # detected corners
        ids                 # marker IDs (shown as text)
    )

show_bgr(annotated_builtin, 'cv2.aruco.drawDetectedMarkers() — built-in function')

## 5. Understanding Rejected Candidates

The `rejected` list contains regions that passed **shape tests** (square-ish, correct border) but **failed ID validation** (the bit pattern didn't match any dictionary entry).

```
ArUco Detection Pipeline:

  Input frame
       │
       ▼
  Adaptive threshold → binary image
       │
       ▼
  Find contours → filter by area and shape
       │
       ▼
  Find squares (4-corner polygons) → candidates
       │
       ▼
  Check border (must be black) → reject non-markers
       │              ← "rejected" list comes from here
       ▼
  Read bit pattern → look up in dictionary
       │
       ▼
  Match found → add to `corners` and `ids`
```

**Why look at rejected candidates?**
- If your marker appears in `rejected` but not in `ids`, your dictionary might be wrong
- Useful for debugging detection failures at distance

## 6. Tuning DetectorParameters

`DetectorParameters` controls every step of the detection pipeline.  
Most of the time defaults work fine, but for challenging conditions (far away markers, motion blur, extreme lighting) you may need to tune.

Key parameters:

| Parameter | Default | Effect |
|---|---|---|
| `adaptiveThreshWinSizeMin` | 3 | Minimum window for adaptive thresholding |
| `adaptiveThreshWinSizeMax` | 23 | Maximum window — increase for larger markers |
| `minMarkerPerimeterRate` | 0.03 | Min marker size relative to image — decrease to detect tiny markers |
| `maxMarkerPerimeterRate` | 4.0 | Max marker size |
| `polygonalApproxAccuracyRate` | 0.05 | Corner approximation tolerance |
| `cornerRefinementMethod` | NONE | Sub-pixel corner refinement (use CORNER_REFINE_SUBPIX for accuracy) |

In [ ]:
# Example: enable sub-pixel corner refinement
# This improves corner accuracy — critical for pose estimation
try:
    # New API
    params = cv2.aruco.DetectorParameters()
    params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
    
    d = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector_refined = cv2.aruco.ArucoDetector(d, params)
    
    gray = cv2.cvtColor(test_scene, cv2.COLOR_BGR2GRAY)
    corners_r, ids_r, rejected_r = detector_refined.detectMarkers(gray)
except AttributeError:
    # Old API
    d = cv2.aruco.Dictionary_get(cv2.aruco.DICT_4X4_50)
    params = cv2.aruco.DetectorParameters_create()
    params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
    gray = cv2.cvtColor(test_scene, cv2.COLOR_BGR2GRAY)
    corners_r, ids_r, rejected_r = cv2.aruco.detectMarkers(gray, d, parameters=params)

n = len(ids_r.flatten()) if ids_r is not None else 0
print(f'With sub-pixel refinement: {n} markers detected')
if ids_r is not None and ids is not None:
    for i in range(min(len(corners), len(corners_r))):
        orig_pt = corners[i].reshape(4,2)[0]
        refin_pt = corners_r[i].reshape(4,2)[0]
        print(f'  Marker {i} TL corner: original={orig_pt}, refined={refin_pt}')
        diff = np.linalg.norm(orig_pt - refin_pt)
        print(f'   -> Sub-pixel shift: {diff:.3f} pixels')

## 7. Detection Failure Modes

Understanding why detection fails is essential for production deployment.

In [ ]:
# Create versions of the scene with different failure conditions
def simulate_failure(scene, mode):
    """Simulate common detection failure conditions."""
    if mode == 'blur':
        return cv2.GaussianBlur(scene, (21, 21), 0)
    elif mode == 'dark':
        return (scene.astype(np.float32) * 0.2).astype(np.uint8)
    elif mode == 'bright':
        return np.clip(scene.astype(np.int32) + 150, 0, 255).astype(np.uint8)
    elif mode == 'noise':
        noise = np.random.normal(0, 40, scene.shape).astype(np.int16)
        return np.clip(scene.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return scene

failure_modes = ['blur', 'dark', 'bright', 'noise']

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, mode in zip(axes.flatten(), failure_modes):
    degraded = simulate_failure(test_scene, mode)
    c, i, r = detect_markers(degraded, api_info)
    n = len(i.flatten()) if i is not None else 0
    result = draw_detections(degraded, c, i, r, draw_rejected=False)
    ax.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Mode: {mode} | Detected: {n}/3 markers')
    ax.axis('off')
plt.suptitle('Detection under different failure conditions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Debug Checklist

If markers are not detected:

1. **Wrong dictionary?** — Must match what was used to generate the marker  
2. **Too dark / too bright?** — Adaptive threshold fails at extremes; add a light source  
3. **Motion blur?** — Use shorter exposure time or reduce robot speed near station  
4. **Too small in frame?** — Decrease `minMarkerPerimeterRate` or move camera closer  
5. **Partial occlusion?** — 50%+ of marker must be visible  
6. **Heavy noise?** — Increase `adaptiveThreshWinSizeMax`  
7. **Appears in `rejected`?** — Dictionary mismatch or bit errors from print quality

## 8. Real-Time Webcam Detection (Script Template)

The cell below is a **complete real-time detection script**.  
It will not run in a notebook (no `imshow` in Colab/notebook environments), but copy it to a `.py` file and run it locally.

Press `Q` to quit.

In [ ]:
# The real-time detection script lives in scripts/aruco/detect_aruco_live.py
# Run it from the repo root in a terminal:
#
#   python scripts/aruco/detect_aruco_live.py
#
# Optional arguments:
#   --dict      NAME    ArUco dictionary (default: DICT_4X4_50)
#   --camera    INT     webcam index (default: 0)
#   --width     INT     capture width (default: 1280)
#   --height    INT     capture height (default: 720)
#   --rejected          show rejected candidates in red (useful for debugging)
#
# Controls while running:
#   Q / Esc   → quit
#
# Shows FPS and detected marker count in real time.

print("See scripts/aruco/detect_aruco_live.py — run from repo root in a terminal.")

## Recap

| Concept | Key Point |
|---|---|
| `corners` | List of (1,4,2) arrays — 4 corner points per marker |
| `ids` | (N,1) int array — ID per marker, or `None` if nothing found |
| `rejected` | Candidates that failed bit-pattern validation |
| Corner order | TL → TR → BR → BL (always, even after rotation) |
| Grayscale input | Detection works on grayscale — convert BGR first |
| `CORNER_REFINE_SUBPIX` | Sub-pixel accuracy — use before pose estimation |
| Dictionary must match | Generation and detection must use identical dictionary |

**Next notebook:** We use the detected corners + calibration to compute 6D pose (rvec, tvec).

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Detect from a saved PNG
# ============================================================
# Load the marker PNG saved in notebook 12 (4x4_id001.png from
# ../assets/aruco_markers/). Run detection on it.
# Print the detected ID and corner coordinates.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img = cv2.imread('../assets/aruco_markers/4x4_id001.png')
# if img is None:
#     # Fallback: generate it
#     d = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
#     m = cv2.aruco.generateImageMarker(d, 1, 500)
#     img = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
# c, i, r = detect_markers(img, api_info)
# if i is not None:
#     print(f'Detected ID: {i.flatten()[0]}')
#     print(f'Corners: {c[0].reshape(4,2)}')
# else:
#     print('Nothing detected')

In [ ]:
# ============================================================
# EXERCISE 2: Draw a crosshair at each marker center
# ============================================================
# Starting from the test_scene, detect markers and draw:
# - A filled circle at the center of each marker
# - Two lines (horizontal and vertical) forming a crosshair
# Display the result.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# output = test_scene.copy()
# c, i, r = detect_markers(test_scene, api_info)
# if i is not None:
#     for corner_set in c:
#         pts = corner_set.reshape(4, 2).astype(int)
#         cx = int(pts[:, 0].mean())
#         cy = int(pts[:, 1].mean())
#         cv2.circle(output, (cx, cy), 8, (0, 255, 0), -1)
#         cv2.line(output, (cx-20, cy), (cx+20, cy), (0, 255, 0), 2)
#         cv2.line(output, (cx, cy-20), (cx, cy+20), (0, 255, 0), 2)
# show_bgr(output, 'Crosshair at marker centers')

In [ ]:
# ============================================================
# EXERCISE 3: Try the wrong dictionary
# ============================================================
# Create a detector using DICT_5X5_100 (wrong dictionary).
# Run it on test_scene which contains DICT_4X4_50 markers.
# How many markers are detected? What's in rejected?
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# wrong_api = make_detector(cv2.aruco.DICT_5X5_100)
# c, i, r = detect_markers(test_scene, wrong_api)
# n = len(i.flatten()) if i is not None else 0
# print(f'Detected with wrong dictionary: {n} markers')
# print(f'Rejected candidates: {len(r)}')
# print('The 4x4 markers appear in rejected because they have the right shape')
# print('but their bit pattern does not match any 5x5 dictionary entry.')